In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedKFold, 
    cross_validate,
    RandomizedSearchCV
)
from sklearn.metrics import (
    confusion_matrix,
    precision_score, 
    recall_score, 
    f1_score, 
    average_precision_score,
    accuracy_score,
    roc_auc_score
)
from xgboost import XGBClassifier



In [ ]:
df = pd.read_csv("../data/depression_data.csv")

## The Basics

In [ ]:
# Display some basic info of the df like number of rows, columns
# the dataframe head, and descriptive info of numerical columns
print(df.shape)
display(df.head())
display(df.describe().T)

A small selection of features here with only a few numerical columns. I'll be treating this as a classification problem as the "history of mental illness" that I want to predict is a yes/no column.   
The info from the kaggle site on the dataset suggested that this data was already relatively clean and deduped but worth checking for safety:

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
# Inspect the target counts
df["History of Mental Illness"].value_counts(dropna=False)

In [ ]:
# Show target graphically - predominantly "no"
sns.countplot(data=df, x="History of Mental Illness")

In [ ]:
# Check the proportions of yes / no in the target
df["History of Mental Illness"].value_counts(normalize=True)

## Numeric Columns

In [ ]:
# Select the numerical columns
numeric_cols = list(df.select_dtypes(include=np.number).columns)

In [ ]:
# Plot histograms of each of the numeric features
df[numeric_cols].hist(figsize=(15,12))

In [ ]:
# Print out table of statistical values
print(
    f"{'Feature':<18} | {'Mean':<8} | "
    f"{'Median':<8} | {'Standard Deviation':<8}"
)
for column in numeric_cols:
    print(
        f"{column:<18} | {df[column].mean():<8.2f} | "
        f"{df[column].median():<8.2f} | {df[column].std():<8.2f}"
    )

##### Visually inspect numerical feature distributions

In [ ]:
sns.boxplot(data=df,
            x="History of Mental Illness",
            y="Age")

In [ ]:
sns.boxplot(data=df,
            x="History of Mental Illness",
            y="Number of Children")

In [ ]:
sns.boxplot(data=df,
            x="History of Mental Illness",
            y="Income")

Some small differences in age, very similar distributions by number of children, very different shapes when considering income. 
To me this suggests that there is some correlation with income, where higher income customers may be less likely to havea a history of mental illness.  
This is not necessarily causal (e.g. a low income job may be unlikely to result in mental illness, but those without mental illnesses may find it easier to secure a higher salary position)

## Categorical Columns

In [ ]:
# Select categorical columns
categorical_cols = list(set(df.columns)-set(numeric_cols))
categorical_cols.remove("Name")
categorical_cols.remove("History of Mental Illness")

In [ ]:
print(categorical_cols)

##### Visually inspect categorical feature distributions

In [ ]:
pd.crosstab(
    df["History of Substance Abuse"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Marital Status"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Education Level"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Family History of Depression"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Alcohol Consumption"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Smoking Status"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Physical Activity Level"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Chronic Medical Conditions"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Sleep Patterns"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Dietary Habits"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

In [ ]:
pd.crosstab(
    df["Employment Status"],
    df["History of Mental Illness"],
    normalize="index"
).plot(kind="bar")

These distributions suggest some small correlation between a history of mental health issues and:
 * Marital status
 * Alcohol consumption   

and larger correlation between a history of mental health issues and:
 * Sleep patterns 
 * Dietary habits
 * Employment status

## Statistical Tests

In [ ]:
# We only have 3 numerical columns, but let's look at the correlation matrix
sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1
)

This correlation matrix is pretty much as I would have expected it:   
 * Age and Number of Children are positively correlated, so people are more likely to have had a greater number of children as they age - this feels rational
 * Number of Children and Income are positively correlated, so people with more income are potentially better suited to support the needs of more children - this also feels rational
 * Income and Age are slightly negatively correlated, so people are likely to be lower income as they age - this was initially surprising to see

My initial expectation was that Income would increase with age, as younger people might be in education or less senior roles which I would expect to pay less. 
On refelction there are legitimate reasons for how this could be true (older people prioritising family over high career salaries, or people who have retired sitting on a lower fixed income resulting in drawing from a pension).   
Let's review the shape of the distribution to see if it fits one or both of these hypotheses. 

In [ ]:
# Group people by their age and calculate each age's mean income
age_income_df = df.groupby("Age").agg(
    mean_income=("Income", "mean")
).reset_index()

# Plot the shape of the resulting distribution
sns.lineplot(
    x=age_income_df["Age"],
    y=age_income_df["mean_income"]
)

This distribution seems to fit both hypotheses on quick inspection: people start out at lower salaries as they enter the workforce, climbing until they peak around the 30-40 range. There is a clear drop off as people enter their 40's (potentially due to family commitments or seeking a better work life balance), and another clear decrease as customers enter their 60's (presumably due to entering retirement).   
These assumptions are likely not the entire story, but are enough that my curiousity is satisfied for the purpose of this exercise. I will likely include all three of these features in the model, as they all seem to behave as expected and none correlate strongly enough to be proxies for each other. 

Before selecting the final features to be used in modelling, I want to look at the p values of each of the features. 

In [ ]:
# Calculate the p_value for each of the categorical features
for column in categorical_cols:
    table = pd.crosstab(
        df[column],
        df["History of Mental Illness"]
    )

    chi2, p, dof, expected = chi2_contingency(table)

    print(f"Column {column}, p: {p:.6f}")

The p values of each categorical feature suggest that the relationship between the target and the `History of Substance Abuse` feature is considerably more likely than the other features to occur through random chance.   
Considering the typical p value limit is around 0.05, I will likely not include this column in my model. 

Additionally, while there are no issues with the p value of the `Chronic Medical Conditions` column I have reservations against using it in the model: it's correlation to the target seems minimal, and in a real world scenario I would worry about the potential bias that may arise based on the nature of these conditions.

In [ ]:
categorical_cols.remove("Chronic Medical Conditions")

## Train Test Split

In [ ]:
feature_cols = numeric_cols + categorical_cols

In [ ]:
# Split the dataset into 80% train and 20% test
# Also cast the target variable into a numeric form for convenience later on
X_train, X_test, y_train, y_test = train_test_split(
    df[feature_cols],
    df["History of Mental Illness"].apply(
        lambda x: 1 if x == "Yes" else 0
    ),
    test_size = 0.2,
    stratify=df["History of Mental Illness"].apply(
    lambda x: 1 if x == "Yes" else 0
    ),
    random_state=42
)

## Sklearn Pipeline

In [ ]:
# Only need this numeric scaler for the regression model
numeric_transformer = Pipeline([
    ("scaler", StandardScaler())
])

# Need to one hot encode our categorical features for all models
categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Create one preprocessor for specifically the regression case
scaled_preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_cols),
    ("categorical", categorical_transformer, categorical_cols)
])

# Create a second preprocessor that does not scale numeric values
preprocessor = ColumnTransformer([
    ("categorical", categorical_transformer, categorical_cols)
])

## Modelling

In [ ]:
# 1st model: A dummy model that always predicts "No"
dummy = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

dummy.fit(X_train, y_train)

#dummy_pred = dummy.predict(X_test)

In [ ]:
# 2nd model: A simple linear regression model
logistic = Pipeline([
    ("preprocessor", scaled_preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logistic.fit(X_train, y_train)

#logistic_pred = logistic.predict(X_test)


In [ ]:
# 3rd model: A random forest 
random_forest = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42
    ))
])

random_forest.fit(X_train, y_train)

#random_forest_pred = random_forest.predict(X_test)

Three pipelines for models have been created, each with increasing complexity. In order to evaluate the performance of each model on our test data, I'll evaluate several performance metrics using cross validation. The idea here is to subsample the training data (reserving a different evaluation batch each time) multiple times, and average the results in order to reduce the effects of any overfitting.

In [ ]:
# Create a dictionary of models to iteratve over
models = {
    "dummy": dummy,
    "logistic": logistic,
    "random_forest": random_forest
}

# Set up our cross validation folds preserving the target class percentage
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Create a list of scoring metrics to evaluate
scoring = ["accuracy", 
           "precision",
           "recall",
           "f1",
           "roc_auc"
]

cv_results = {}

# Loop over the models and save the result of each score on each fold
# We will take the mean of these later
for model in tqdm(models.keys()):
    cv_results[model] = cross_validate(
        models[model],
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

In [ ]:
def print_model_scores(models, cv_results):
    # Print a formatted table of model performance metrics
    print(
        f"{"Model":<13} | "
        f"{"Accuracy":<8} | "
        f"{"Precision":<9} | "
        f"{"Recall":<6} | "
        f"{"F1 Score":<8} | "
        f"{"ROC-AUC"}"
    )
    for model in models.keys():
        print(
            f"{model:<13} | "
            f"{cv_results[model]["test_accuracy"].mean():<8.4f} | "
            f"{cv_results[model]["test_precision"].mean():<9.4f} | "
            f"{cv_results[model]["test_recall"].mean():<6.4f} | "
            f"{cv_results[model]["test_f1"].mean():<8.4f} | "
            f"{cv_results[model]["test_roc_auc"].mean():.4f}"
        )

In [ ]:
print_model_scores(models, cv_results)

Looking at these models produces a pretty clear result. The dummy model always predicts that an individual does not have a history of mental health issues (as this is the most likely scenario). This results in an accuracy of almost 70%, but in practicality this model is useless.   
Interestingly, the logistic regression model has exactly the same accuracy as the dummy model even though it's ROC-AUC score is improved. This implies that there is some modelling happening in the logistic case, but due to the nature of the classification problem it is not having any meaningful effect (more on this later).   
The random forest model has a more interesting result, although the precision score suggests the number of false positives is quite high. The recall indicates that the model was able to identify 55% of people in the data who had a history of mental health problems, which is considerably better than either of the other models. 

I'm going to try and XGBClassifier now to see if we can't beat this result. 

In [ ]:
# Create pipeline for hyperparameter tuning
xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        eval_metric="logloss",
        random_state=42
    ))
])

In [ ]:
# Dictionary of hyperparameters to evaluate
param_grid = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__gamma": [0, 0.1, 0.5, 1, 2],
    "model__reg_alpha": [0, 0.01, 0.1, 1],
    "model__reg_lambda": [0.1, 1, 5, 10],
}

In [ ]:
# RandomizedSearchCV performed to save time as this is not quick
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=25,
    scoring="roc_auc",
    cv=cv,
    verbose=1,
    random_state=42
)

best_params = {
    'subsample': 0.7, 
    'reg_lambda': 10, 
    'reg_alpha': 1, 
    'n_estimators': 300, 
    'min_child_weight': 5, 
    'max_depth': 4, 
    'learning_rate': 0.03, 
    'gamma': 2, 
    'colsample_bytree': 0.7
}

new_search = False

if new_search:
    random_search.fit(X_train, y_train)
    print(random_search.best_params_)
    print(random_search.best_score_)

In [ ]:
# Unpack the optimal parameters into a new pipeline for modelling
xgb_optimised = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(**best_params))
])

xgb_optimised.fit(X_train, y_train)

In [ ]:
# Now to evaluate the performance of the xgboost model
models["xgb"] = xgb_optimised

cv_results["xgb"] = cross_validate(
        models["xgb"],
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

In [ ]:
print_model_scores(models, cv_results)

Practically, the xgboost model performs identically to the dummy and logistic models (essentially always predicting that there is no history of mental illness) despite the best ROC-AUC score yet. This indicates that there is some succesful modelling taking place, but why is this not coming through as successful predictions?

In [ ]:
# Look in detail at the xgb models prediction of how likely each person is to have a history of mental illness
y_prob = xgb_optimised.predict_proba(X_test)[:, 1]

print(f"{"Min":<6} | {y_prob.min():.4f}")
print(f"{"Max":<6} | {y_prob.max():.4f}")
print(f"{"Mean":<6} | {y_prob.mean():.4f}")
print(f"{"PR-AUC":<6} | {average_precision_score(y_test, y_prob):.4f}")

print()

print(
    f"{"Threshold":<9} | "
    f"{"Precision":<9} | "
    f"{"Recall":<6} | "
    f"{"F1 Score":<8} | "
)

# Instead of deciding that the model has predicted a positive result when probability > 0.5, 
# look at how the results change at different thresholds 
for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    precision = precision_score(y_test, 
                                y_pred_threshold, 
                                zero_division=0)
    recall = recall_score(y_test, 
                          y_pred_threshold, 
                          zero_division=0)
    f1 = f1_score(y_test, 
                  y_pred_threshold, 
                  zero_division=0)
    
    print(
        f"{threshold:<9} | {precision:<9.4f} | "
        f"{recall:<6.4f} | {f1:<8.4f} | "
    )



The numbers above show that our xgboost model has low confidence in its predictions: even when the model is maximally confident that someone has a history of mental illness, they only give that person a 41.71% likelihood. If we optimise the threshold for a positive result, the model begins to become more predictive.   
This can certainly be optimised more intelligently, but for the sake of this exercise I'm going to take a look at performance at a threshold of 0.3. 

## Confusion Matrices

In [ ]:
y_pred = (dummy.predict_proba(X_test)[:, 1] > 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Labels')
plt.ylabel('True Lables')
plt.title('Dummy Model Confusion Matrix 0.5')
plt.show()

In [ ]:
y_pred = (dummy.predict_proba(X_test)[:, 1] > 0.3).astype(int)
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Labels')
plt.ylabel('True Lables')
plt.title('Dummy Model Confusion Matrix 0.3')
plt.show()

The dummy model always predicts a "No" result as expected - because the dataset being used is predominantly "No" results this does provide a good accuracy, but this model is only used here as a baseline. 

In [ ]:
y_pred = (logistic.predict_proba(X_test)[:, 1] > 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Labels')
plt.ylabel('True Lables')
plt.title('Logistic Regression Confusion Matrix 0.5')
plt.show()

In [ ]:
y_pred = (logistic.predict_proba(X_test)[:, 1] > 0.3).astype(int)
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Labels')
plt.ylabel('True Lables')
plt.title('Logistic Regression Confusion Matrix 0.3')
plt.show()

The logistic regression model does better than simply always predicting a "No" result at the 0.3 threshold, but the number of false positives and false negatives are still very high. The model is far more likely to predict that a person may have a history of mental illness, but this means it is also significantly more likely to return a false positive result. 

In [ ]:
y_pred = (random_forest.predict_proba(X_test)[:, 1] > 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Labels')
plt.ylabel('True Lables')
plt.title('Random Forest Confusion Matrix 0.5')
plt.show()

In [ ]:
y_pred = (random_forest.predict_proba(X_test)[:, 1] > 0.3).astype(int)
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Labels')
plt.ylabel('True Lables')
plt.title('Random Forest Confusion Matrix 0.3')
plt.show()

The random forest model is clearly the best performing model at the 0.5 threshold, as none of the others are even able to identify a single positive result. There are still a high proportion of false positives present in the result, but this may be preferable to the alternative of no predictions whatsoever. 
At the 0.3 threshold, this model becomes overconfident and predicts almost every result as a positive. 

In [ ]:
y_pred = (xgb_optimised.predict_proba(X_test)[:, 1] > 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Labels')
plt.ylabel('True Lables')
plt.title('XGBoost Confusion Matrix 0.5')
plt.show()

In [ ]:
y_pred = (xgb_optimised.predict_proba(X_test)[:, 1] > 0.3).astype(int)
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Labels')
plt.ylabel('True Lables')
plt.title('XGBoost Confusion Matrix 0.3')
plt.show()

The XGBoost model at the 0.3 threshold does seem to be the best result out of any viewed - but only just. The result of the random forest model at the 0.5 threshold is only slightly worse than the xgboost model at 0.3. There is an argument that with more time the XGBoost model could be tuned further to provide a better result, but in this exercise the difference between the two is minimal. 

## Final Model Comparison

In [ ]:
def evaluate_at_threshold(model, X, y, cv, threshold=0.3):
    y_prob = cross_val_predict(
        model,
        X,
        y,
        cv=cv,
        method="predict_proba"
    )[:, 1]

    y_pred = (y_prob > threshold).astype(int)

    return {
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_prob)
    }

In [ ]:
low_threshold_cv_results = {}

for model in tqdm(models.keys()):
    low_threshold_cv_results[model] = evaluate_at_threshold(
        models[model],
        X_train,
        y_train,
        cv=cv
    )

In [ ]:
# Reprint our earlier metrics at the 0.3 threshold
print(
    f"{"Model":<13} | "
    f"{"Accuracy":<8} | "
    f"{"Precision":<9} | "
    f"{"Recall":<6} | "
    f"{"F1 Score":<8} | "
    f"{"ROC AUC"}"
)
for model in models.keys():
    print(
        f"{model:<13} | "
        f"{low_threshold_cv_results[model]["accuracy"]:<8.4f} | "
        f"{low_threshold_cv_results[model]["precision"]:<9.4f} | "
        f"{low_threshold_cv_results[model]["recall"]:<6.4f} | "
        f"{low_threshold_cv_results[model]["f1"]:<8.4f} | "
        f"{low_threshold_cv_results[model]["roc_auc"]:.4f}"
    )

`random_forest | 0.5576   | 0.3537    | 0.5500 | 0.4305   | 0.5687`

Included above for posterity is the result of the random forest model evaluation at the 0.5 threshold. In a real life scenario I think these results would warrant further discussion - it all depends on the desired final output of the model and how this would feed into the wider business. There may be advantages in either case that would help decide which is the best model to use going forward.   
In reality, this would probably result in several meetings to discuss this. For the sake of this exercise, the results are very similar so I'll take the one with the slightly better scores. I'll move forward with the XGBoost model just to save some time spend debating and deep in thought. 